Thin wrapper: no logic here. Set `REPO_URL` to your GitHub repository (private repos: add a Kaggle secret `GITHUB_TOKEN`). Enable a GPU accelerator and internet.

In [ ]:
REPO_URL = 'https://github.com/<you>/<repo>.git'  # edit
import os, subprocess
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret('GITHUB_TOKEN')
    REPO_URL = REPO_URL.replace('https://', f'https://{tok}@')
except Exception:
    pass
if not os.path.isdir('/kaggle/working/repo'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, '/kaggle/working/repo'], check=True)
%cd /kaggle/working/repo
!pip install -q -e .[metrics]

Train with checkpoints every 30 min to `/kaggle/working/run_*` (the notebook output). If the session is cut off, re-run with the previous output attached and copied back: `--resume` continues from `last.pt`. `PAIRS` is the attached dataset of shards; leave it empty to simulate on the fly.

In [ ]:
MODEL = 'unet'   # unet | rtlvit | vit5
PAIRS = '/kaggle/input/sodacan-pairs'  # attached dataset from kaggle_simulate.ipynb
PREVIOUS = ''  # e.g. '/kaggle/input/sodacan-run-unet' to resume from an earlier session
import os, shutil
out = f'/kaggle/working/run_{MODEL}'
if PREVIOUS and not os.path.exists(out):
    shutil.copytree(PREVIOUS, out)

In [ ]:
!python -m sodacan.restore.train --config configs/train_{MODEL}.yaml --out {out} --resume --max-minutes 690 {'--shards ' + PAIRS if PAIRS else ''}

In [ ]:
!python scripts/evaluate_restore.py --models {out}/best.pt --pairs 64 --out {out}